In [1]:
import os
import json
doc_str = "../data/validated_examples_RETRY.json"
if not os.path.exists(doc_str):
    print("prob with path")
else:
    with open(doc_str, 'r', encoding='utf-8') as f:
        retry_passed_prob = json.load(f)
print(f"len of prob object is: {len(retry_passed_prob)}")


len of prob object is: 67


In [2]:
from collections import Counter
retry_passed_prob[0].keys()
diff= {}
for el in retry_passed_prob:
    if el["difficulty"] == "Easy":
        diff["Easy"] = diff.get("Easy", 0) +1
    elif el["difficulty"] == "Medium":
        diff['Medium'] = diff.get("Medium", 0) +1
    else:
        diff["Hard"] = diff.get("Hard", 0) +1
print(diff.items())

dict_items([('Easy', 16), ('Medium', 24), ('Hard', 27)])


In [3]:
from dotenv import load_dotenv
load_dotenv("../.env")
import os

token = os.environ.get("HF_TOKEN")
print(f"Token is: {token}")

Token is: REDACTED_HF_TOKEN


In [2]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

# define the quantization
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    "AndreiSobo/pact-qwen-tutor",
    quantization_config = quant_config,
    torch_dtype="auto",
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("AndreiSobo/pact-qwen-tutor")

# Example conversation (matching training format)
messages = [
    {
        "role": "system", 
        "content": "You are PACT, a Socratic Python coding tutor. Help students learn through guided questions and hints, not direct answers."
    },
    {
        "role": "user", 
        "content": """Problem: Two Sum

Given an array of integers nums and an integer target, return indices of the two numbers that add up to target.

Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [2,1]

Constraints:
- 2 <= nums.length <= 104
- -109 <= nums[i] <= 109
- Only one valid answer exists.

My code:
```python
def twoSum(nums, target):
    for i in range(len(nums)):
        for j in range(len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
```

It runs but gives wrong output for some test cases.

Can you give me a hint?"""
    }
]

# Generate response
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=200,
    temperature=0.7,
    do_sample=True,
    top_p=0.9
)

response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

ValueError: Some modules are dispatched on the CPU or the disk. Make sure you have enough GPU RAM to fit the quantized model. If you want to dispatch the model on the CPU or the disk while keeping these modules in 32-bit, you need to set `load_in_8bit_fp32_cpu_offload=True` and pass a custom `device_map` to `from_pretrained`. Check https://huggingface.co/docs/transformers/main/en/main_classes/quantization#offload-between-cpu-and-gpu for more details. 

In [3]:
import gc
import torch

# Delete the variables
if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer

# Force Python garbage collection
gc.collect()

# Clear the NVidia VRAM cache
torch.cuda.empty_cache()

In [1]:
# Only run this once!
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    "AndreiSobo/pact-qwen-tutor",
    quantization_config=quant_config,
    device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained("AndreiSobo/pact-qwen-tutor")

/home/andrei/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading checkpoint shards: 100%|██████████| 4/4 [00:25<00:00,  6.31s/it]


In [2]:
messages = [
    {
        "role": "system", 
        "content": "You are PACT, a Socratic Python coding tutor. Help students learn through guided questions and hints, not direct answers."
    },
    {
        "role": "user", 
        "content": """Problem: Two Sum

Given an array of integers nums and an integer target, return indices of the two numbers that add up to target.

Example 1:

Input: nums = [2,7,11,15], target = 9
Output: [0,1]
Explanation: Because nums[0] + nums[1] == 9, we return [0, 1].

Example 2:

Input: nums = [3,2,4], target = 6
Output: [2,1]

Constraints:
- 2 <= nums.length <= 104
- -109 <= nums[i] <= 109
- Only one valid answer exists.

My code:
```python
def twoSum(nums, target):
    for i in range(len(nums)):
        for j in range(len(nums)):
            if nums[i] + nums[j] == target:
                return [i, j]
```

It runs but gives wrong output for some test cases.

Can you give me a hint?"""
    }
]
text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(text, return_tensors="pt").to(model.device)

outputs = model.generate(**inputs, max_new_tokens=200)
response = tokenizer.decode(outputs[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
print(response)

Starting from v4.46, the `logits` model output will have the same type as the model (except at train time, where it will always be FP32)


When you're iterating over pairs of elements from your input list, how many times should each element be checked against every other element? Think about what happens when an element is compared with itself. Does that produce a valid pair according to the problem statement?
